# Generators & Decorators

## Part A — `yield` & Generators
1. The `yield` Keyword
2. Generator Functions
3. Generator Expressions
4. Stateful Generators & `send()`
5. Chaining Generators with `yield from`

## Part B — Decorators
6. Functions as First-Class Objects (the foundation)
7. Writing Your First Decorator
8. Decorators with Arguments
9. Stacking Decorators
10. Real-World Decorator Patterns


# PART A — `yield` & Generators

## 1. The `yield` Keyword

### The problem `yield` solves

Imagine you want to produce a sequence of 1 million numbers. A regular function would build the **entire list in memory** before returning anything. That's wasteful.

`yield` lets a function **pause execution** and hand a value to the caller — then **resume exactly where it left off** the next time it's called.

Think of it like a vending machine: it gives you one item, waits, gives you another — it doesn't pour everything out at once.

| `return` | `yield` |
|---|---|
| Exits the function permanently | Pauses the function, saves state |
| Returns one value | Can produce many values over time |
| The function is "done" | The function can be resumed |
| Returns any object | Turns the function into a **generator** |

In [ ]:
# Example 1: Your very first yield

def simple_generator():
    print("Step 1: about to yield 10")
    yield 10
    print("Step 2: about to yield 20")
    yield 20
    print("Step 3: about to yield 30")
    yield 30
    print("Step 4: function is done")

# Calling the function returns a generator object — nothing runs yet!
gen = simple_generator()
print(type(gen))   # <class 'generator'>
print()

# next() advances the generator to the next yield
value = next(gen)
print(f"Got: {value}\n")

value = next(gen)
print(f"Got: {value}\n")

value = next(gen)
print(f"Got: {value}\n")

In [ ]:
# Example 2: Iterating a generator with a for loop
# A for loop calls next() automatically until StopIteration is raised.

def countdown(n):
    while n > 0:
        yield n
        n -= 1

for number in countdown(5):
    print(number, end=" ")  # 5 4 3 2 1

In [ ]:
# Example 3: Memory efficiency — compare list vs generator
import sys

# A list materialises ALL values immediately
big_list = [x * 2 for x in range(100_000)]

# A generator produces values one at a time, on demand
def big_gen(n):
    for x in range(n):
        yield x * 2

gen = big_gen(100_000)

print(f"List size in memory : {sys.getsizeof(big_list):,} bytes")
print(f"Generator size      : {sys.getsizeof(gen):,} bytes")
print("\nThe generator is tiny regardless of how many items it produces!")

### Exercises — `yield` Basics

In [ ]:
# Exercise 1.1
# Write a generator function `first_n_evens(n)` that yields
# the first n even numbers starting from 0 (i.e. 0, 2, 4, 6, ...).
#
# Use a for loop to print all values from first_n_evens(6).
# Expected output:  0  2  4  6  8  10

# YOUR CODE HERE


In [ ]:
# Exercise 1.2
# Write a generator function `squares(start, stop)` that yields
# the square of each integer from start up to (but not including) stop.
#
# Collect all values into a list using: list(squares(1, 6))
# Expected output: [1, 4, 9, 16, 25]
#
# Then manually call next() three times on a new squares(10, 20) generator
# and print each result.

# YOUR CODE HERE


In [ ]:
# Exercise 1.3
# Write a generator `fibonacci()` that yields an INFINITE sequence
# of Fibonacci numbers (0, 1, 1, 2, 3, 5, 8, 13, ...).
# (Don't worry — it won't run forever because we'll stop it.)
#
# Use it to print the first 10 Fibonacci numbers.
# Hint: Use a counter in a for loop, or itertools.islice.
#
# Expected output: 0 1 1 2 3 5 8 13 21 34

# YOUR CODE HERE


## 2. Generator Functions — Patterns & Power

Generator functions can do everything regular functions do: take arguments, use loops, conditionals, try/except, and even `return` (which raises `StopIteration` with an optional value).

They're perfect for:
- **Lazy pipelines** — process data without loading it all into memory
- **Infinite sequences** — numbers, events, streams
- **Custom iterators** — without writing a class with `__iter__`/`__next__`

In [ ]:
# Example 1: Filtering inside a generator

def primes_up_to(limit):
    """Yield prime numbers up to limit."""
    def is_prime(n):
        if n < 2:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True

    for n in range(2, limit + 1):
        if is_prime(n):
            yield n

print(list(primes_up_to(30)))  # [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

In [ ]:
# Example 2: Reading a large file line by line (lazy I/O pattern)
# We'll simulate a large file using a list of lines.

def read_lines(lines):
    """Yield non-empty, stripped lines — one at a time."""
    for line in lines:
        stripped = line.strip()
        if stripped:          # skip blank lines
            yield stripped

simulated_file = [
    "  Hello World  \n",
    "\n",
    "  Python is great  ",
    "",
    "  Generators save memory  \n",
]

for line in read_lines(simulated_file):
    print(repr(line))

In [ ]:
# Example 3: Generator as a lazy data pipeline

raw_data = ["  Alice,30  ", "  Bob,25  ", "  Charlie,35  ", "  Diana,28  "]

def parse_records(lines):
    for line in lines:
        name, age = line.strip().split(",")
        yield {"name": name.strip(), "age": int(age.strip())}

def filter_adults(records, min_age=30):
    for record in records:
        if record["age"] >= min_age:
            yield record

# Chain generators — nothing is computed until we iterate!
pipeline = filter_adults(parse_records(raw_data), min_age=30)

for person in pipeline:
    print(person)

### Exercises — Generator Functions

In [ ]:
# Exercise 2.1
# Write a generator `running_total(numbers)` that takes an iterable
# of numbers and yields the cumulative (running) sum after each item.
#
# Example: running_total([1, 2, 3, 4, 5])
# Yields:   1, 3, 6, 10, 15

# YOUR CODE HERE


In [ ]:
# Exercise 2.2
# Write a generator `word_lengths(sentence)` that:
#   1. Splits the sentence into words
#   2. Yields (word, length) tuples — but ONLY for words longer than 3 characters
#
# Test with: "The quick brown fox jumps over the lazy dog"
# Expected output (tuples): ('quick', 5) ('brown', 5) ('jumps', 5) ('over', 4) ('lazy', 4)

# YOUR CODE HERE


In [ ]:
# Exercise 2.3
# Build a lazy CSV parser as a generator pipeline.
#
# Given the raw data below, create TWO generators:
#   1. `parse_csv(rows)` — yields each row as a dict {"name", "score", "grade"}
#      where score is an int and grade is a str.
#   2. `passing_students(records)` — yields only students with score >= 60.
#
# Chain them and print each passing student's name and score.

raw_csv = [
    "name,score,grade",
    "Alice,85,B",
    "Bob,52,F",
    "Charlie,91,A",
    "Diana,47,F",
    "Eve,73,C",
    "Frank,60,D",
]

# YOUR CODE HERE


## 3. Generator Expressions

Generator expressions are the **lazy cousin of list comprehensions**. They use `()` instead of `[]` and produce a generator object — no values are computed until you iterate.

```python
# List comprehension — builds the whole list immediately
squares_list = [x**2 for x in range(1000)]

# Generator expression — produces values one at a time
squares_gen  = (x**2 for x in range(1000))
```

Use them anywhere an iterable is expected: `sum()`, `max()`, `list()`, `for` loops, etc.

In [ ]:
# Example 1: Generator expression basics

# Sum of squares without ever building a list
total = sum(x**2 for x in range(1, 11))
print(f"Sum of squares 1-10: {total}")   # 385

# Find the first even square > 50
result = next(x**2 for x in range(1, 100) if x**2 > 50 and x % 2 == 0)
print(f"First even square > 50: {result}")  # 64

# Combine with max()
words = ["apple", "banana", "cherry", "date", "elderberry"]
longest = max(len(w) for w in words)
print(f"Longest word has {longest} characters")

In [ ]:
# Example 2: Chaining generator expressions (pipeline style)

data = range(1, 21)

# Step 1: filter odds
odds = (x for x in data if x % 2 != 0)

# Step 2: square them
squared = (x**2 for x in odds)

# Step 3: keep only those > 50
filtered = (x for x in squared if x > 50)

# Nothing has been computed yet! Only runs when we consume:
print(list(filtered))  # [81, 121, 169, 225, 289, 361]

In [ ]:
# Example 3: When to use list vs generator expression

numbers = range(1_000_000)

# ✅ Use a generator when you only need ONE pass (e.g. sum, any, all)
total = sum(x * 2 for x in numbers)

# ✅ Use a list when you need to:
#   - Access by index
#   - Iterate multiple times
#   - Know the length upfront
first_10_doubled = [x * 2 for x in range(10)]
print(first_10_doubled[3])  # index access — needs a list

# ✅ any() / all() short-circuit — perfect with generators
has_negative = any(x < 0 for x in [1, 3, -1, 5])
all_positive = all(x > 0 for x in [1, 3, 2, 5])
print(f"Has negative: {has_negative}")  # True
print(f"All positive: {all_positive}")  # True

### Exercises — Generator Expressions

In [ ]:
# Exercise 3.1
# Using a SINGLE generator expression (no def, no for loop outside),
# compute the sum of all multiples of 3 or 5 below 1000.
#
# Expected answer: 233168
# (This is Project Euler Problem 1!)

# YOUR CODE HERE


In [ ]:
# Exercise 3.2
# Given this list of sentences, use a generator expression to:
#   1. Convert each sentence to uppercase
#   2. Only include sentences that contain the word "PYTHON" (after uppercasing)
#   3. Print each result
#
# Expected output:
#   PYTHON IS AWESOME
#   I LOVE PYTHON
#   PYTHON GENERATORS ARE COOL

sentences = [
    "Python is awesome",
    "Java is also popular",
    "I love Python",
    "Generators are lazy",
    "Python generators are cool",
]

# YOUR CODE HERE


In [ ]:
# Exercise 3.3
# You have a list of (student_name, scores_list) tuples.
# Using generator expressions (no def required), compute:
#   a) The average score for each student (as a generator of (name, avg) tuples)
#   b) The name of the student with the highest average
#   c) Whether ALL students have at least one score above 70
#
# Hint: you may use max() with a key argument.

students = [
    ("Alice",   [85, 92, 78, 95]),
    ("Bob",     [60, 55, 72, 48]),
    ("Charlie", [91, 88, 94, 97]),
    ("Diana",   [70, 65, 80, 75]),
]

# a) Generator of (name, avg) tuples — print them
# YOUR CODE HERE

# b) Name with highest average
# YOUR CODE HERE

# c) Do all students have at least one score above 70?
# YOUR CODE HERE


**For really motivated students ONLY**
======================================
## 4. Stateful Generators & `send()`

Generators are not just for output — they can **receive values** too, using `send()`.

- `next(gen)` is equivalent to `gen.send(None)` — it resumes and gets the next `yield` value.
- `gen.send(value)` resumes the generator AND passes `value` into the `yield` expression.
- The generator must be **primed** (advanced to the first `yield`) before you can `send()`.
- `gen.close()` stops the generator (raises `GeneratorExit` inside it).

This turns generators into powerful **coroutines** — functions that can receive and produce data iteratively.

In [1]:
# Example 1: send() — two-way communication

def accumulator():
    """Keeps a running total; accepts new values via send()."""
    total = 0
    while True:
        # yield pauses here; the VALUE sent back IS what yield evaluates to
        value = yield total
        if value is None:
            break
        total += value

gen = accumulator()
next(gen)           # Prime: advance to the first yield (total = 0)

print(gen.send(10))  # sends 10 → total becomes 10, yields 10
print(gen.send(25))  # sends 25 → total becomes 35, yields 35
print(gen.send(5))   # sends 5  → total becomes 40, yields 40

10
35
40


In [ ]:
# Example 2: A stateful generator — ID generator

def id_generator(prefix="ID", start=1):
    """Generates unique IDs; reset when you send a new prefix."""
    n = start
    while True:
        new_prefix = yield f"{prefix}-{n:04d}"
        if new_prefix is not None:
            prefix = new_prefix
            n = start
        else:
            n += 1

gen = id_generator("USR")
print(next(gen))           # USR-0001
print(next(gen))           # USR-0002
print(next(gen))           # USR-0003
print(gen.send("PROD"))    # reset to PROD prefix → PROD-0001
print(next(gen))           # PROD-0002

In [ ]:
# Example 3: yield from — delegating to a sub-generator
# `yield from iterable` is shorthand for yielding every item from another iterable.

def flatten(nested):
    """Recursively flatten a nested list using yield from."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # delegate to recursive call
        else:
            yield item

data = [1, [2, 3], [4, [5, 6]], [[7, [8, 9]]]]
print(list(flatten(data)))   # [1, 2, 3, 4, 5, 6, 7, 8, 9]

# yield from also works with any iterable
def combined(*iterables):
    for it in iterables:
        yield from it

print(list(combined([1, 2], "abc", range(3))))

### Exercises — Stateful Generators

In [ ]:
# Exercise 4.1
# Write a generator `counter(start=0, step=1)` that:
#   - Yields incrementing values (start, start+step, start+2*step, ...)
#   - When you send() a number, it RESETS the counter to that number
#   - When you send() None, it just increments normally
#
# Test:
#   gen = counter(0, 2)
#   next(gen)      → 0
#   next(gen)      → 2
#   next(gen)      → 4
#   gen.send(10)   → 10   (reset)
#   next(gen)      → 12
#   next(gen)      → 14

# YOUR CODE HERE


In [ ]:
# Exercise 4.2
# Write a generator `moving_average()` that:
#   - Accepts numbers via send()
#   - After each send, yields the average of ALL numbers received so far
#
# Test:
#   gen = moving_average()
#   next(gen)         # prime the generator
#   gen.send(10)  → 10.0
#   gen.send(20)  → 15.0
#   gen.send(30)  → 20.0
#   gen.send(40)  → 25.0

# YOUR CODE HERE


In [ ]:
# Exercise 4.3
# Use `yield from` to write a generator `chain_ranges(*specs)`
# where each spec is a (start, stop, step) tuple.
# It should yield all values from each range spec in order.
#
# Then write a separate generator `interleave(gen1, gen2)` that
# yields one item from gen1, then one from gen2, alternating
# until BOTH are exhausted.
#
# Test chain_ranges with: (0, 5, 1), (10, 20, 3), (100, 103, 1)
# Expected: [0, 1, 2, 3, 4, 10, 13, 16, 19, 100, 101, 102]
#
# Test interleave([1, 2, 3], ['a', 'b', 'c', 'd'])
# Expected: [1, 'a', 2, 'b', 3, 'c', 'd']

# YOUR CODE HERE


# PART B — Decorators

## 5. Functions as First-Class Objects (The Foundation)

To understand decorators, you first need to know that in Python, **functions are objects** — just like integers or strings. They can be:
- Assigned to variables
- Passed as arguments to other functions
- Returned from functions
- Stored in data structures

A function that accepts or returns another function is called a **higher-order function**.

In [ ]:
# Example 1: Functions are objects

def greet(name):
    return f"Hello, {name}!"

# Assign function to a variable (no parentheses = the function itself, not its result)
say_hello = greet
print(say_hello("Alice"))  # Hello, Alice!
print(type(say_hello))     # <class 'function'>

In [ ]:
# Example 2: Passing functions as arguments

def apply_twice(func, value):
    """Calls func on value, then calls func on the result."""
    return func(func(value))

def double(x):
    return x * 2

def add_ten(x):
    return x + 10

print(apply_twice(double, 3))    # double(double(3)) = double(6) = 12
print(apply_twice(add_ten, 5))   # add_ten(add_ten(5)) = add_ten(15) = 25

In [ ]:
# Example 3: Returning a function — closures
# A closure is a function that "remembers" the variables from its enclosing scope.

def make_multiplier(factor):
    """Returns a NEW function that multiplies by factor."""
    def multiplier(x):          # inner function
        return x * factor       # 'factor' is captured from the outer scope
    return multiplier           # return the function itself

double  = make_multiplier(2)
triple  = make_multiplier(3)
tenx    = make_multiplier(10)

print(double(5))   # 10
print(triple(5))   # 15
print(tenx(5))     # 50

## 6. Writing Your First Decorator

A **decorator** is a function that **wraps another function** to add behaviour before and/or after it runs — without modifying the original function's code.

```
Original function  →  [decorator wraps it]  →  Enhanced function
```

The `@decorator` syntax is just **syntactic sugar** for:
```python
my_function = decorator(my_function)
```

Always use `@functools.wraps(func)` inside your decorator to preserve the original function's name and docstring.

In [ ]:
# Example 1: Building a decorator step by step
import functools

# Step A — a plain wrapper (no @ syntax yet)
def shout(func):
    def wrapper(*args, **kwargs):          # *args/**kwargs: accept any arguments
        result = func(*args, **kwargs)     # call the original function
        return result.upper()             # add behaviour: make it LOUD
    return wrapper

def greet(name):
    return f"Hello, {name}"

greet = shout(greet)          # manual decoration
print(greet("Alice"))         # HELLO, ALICE

In [ ]:
# Example 2: The @ syntax + functools.wraps
import functools

def log_call(func):
    """Decorator: prints a message before and after calling func."""
    @functools.wraps(func)               # preserves original name & docstring
    def wrapper(*args, **kwargs):
        print(f"→ Calling {func.__name__}() with args={args} kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"← {func.__name__}() returned {result!r}")
        return result
    return wrapper

@log_call                               # ← sugar for: add = log_call(add)
def add(a, b):
    """Returns the sum of a and b."""
    return a + b

result = add(3, 4)
print(f"Result: {result}")
print(f"Function name: {add.__name__}")  # 'add' thanks to @wraps
print(f"Docstring: {add.__doc__}")

In [ ]:
# Example 3: A timing decorator
import functools
import time

def timer(func):
    """Measures and prints how long func takes to run."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱  {func.__name__}() took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum integers from 0 to n with a tiny delay."""
    time.sleep(0.05)   # simulate work
    return sum(range(n))

print(slow_sum(1_000_000))

### Exercises — Basic Decorators

In [ ]:
# Exercise 6.1
# Write a decorator `uppercase_result` that converts a function's
# string return value to uppercase.
#
# Apply it to:
#   def greet(name): return f"hello, {name}"
#   def describe(item): return f"this is a {item}"
#
# greet("world")     → "HELLO, WORLD"
# describe("cat")    → "THIS IS A CAT"

import functools

# YOUR CODE HERE


In [ ]:
# Exercise 6.2
# Write a decorator `validate_positive` that raises a ValueError
# if ANY positional argument to the decorated function is <= 0.
#
# Apply it to:
#   def area(width, height): return width * height
#   def volume(l, w, h): return l * w * h
#
# area(3, 4)     → 12
# area(-1, 4)    → ValueError: All arguments must be positive
# volume(2,3,0)  → ValueError: All arguments must be positive

import functools

# YOUR CODE HERE


In [ ]:
# Exercise 6.3
# Write a decorator `count_calls` that tracks how many times
# a function has been called.
#
# The call count should be accessible as: func.call_count
#
# Apply it to a function `roll_dice()` that returns a random int 1-6.
# Call it 5 times and then print: "roll_dice was called 5 times"
#
# Hint: you can set attributes on the wrapper function:
#   wrapper.call_count = 0

import functools
import random

# YOUR CODE HERE


## 7. Decorators with Arguments

Sometimes you want to **configure** a decorator — e.g., `@retry(times=3)` or `@cache(maxsize=128)`.

This requires an **extra layer of nesting**: a function that *returns* a decorator.

```
decorator_factory(args)
    └── returns decorator(func)
                └── returns wrapper(*args, **kwargs)
```

In [ ]:
# Example 1: A decorator factory — repeat(n)
import functools

def repeat(times):
    """Decorator factory: runs the function `times` times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def say(message):
    print(message)

say("Hello!")  # prints Hello! three times

In [ ]:
# Example 2: @retry(max_attempts, exceptions)
import functools
import random

def retry(max_attempts=3, exceptions=(Exception,)):
    """Retry the function up to max_attempts times on specified exceptions."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

call_count = 0

@retry(max_attempts=4, exceptions=(ValueError,))
def unreliable_api():
    """Fails the first 3 times, succeeds on the 4th."""
    global call_count
    call_count += 1
    if call_count < 4:
        raise ValueError("Service temporarily unavailable")
    return "Success!"

print(unreliable_api())

In [ ]:
# Example 3: A simple memoization decorator with max cache size
import functools

def memoize(maxsize=None):
    """Cache function results. maxsize=None means unlimited cache."""
    def decorator(func):
        cache = {}
        @functools.wraps(func)
        def wrapper(*args):
            if args not in cache:
                if maxsize and len(cache) >= maxsize:
                    # Simple eviction: remove oldest entry
                    oldest = next(iter(cache))
                    del cache[oldest]
                cache[args] = func(*args)
            return cache[args]
        wrapper.cache = cache   # expose cache for inspection
        return wrapper
    return decorator

@memoize(maxsize=10)
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print([fib(i) for i in range(10)])  # Fast due to caching
print("Cache:", fib.cache)

### Exercises — Decorators with Arguments

In [ ]:
# Exercise 7.1
# Write a decorator factory `clamp_result(min_val, max_val)` that
# clamps (clips) the numeric return value of the decorated function
# to stay within [min_val, max_val].
#
# @clamp_result(0, 100)
# def score(raw): return raw
#
# score(85)   → 85
# score(120)  → 100
# score(-5)   → 0

import functools

# YOUR CODE HERE


In [ ]:
# Exercise 7.2
# Write a decorator factory `log_level(level)` where level is one
# of "INFO", "WARNING", or "ERROR".
# Before calling the function, print: "[LEVEL] Calling func_name"
# After calling it, print: "[LEVEL] func_name completed"
#
# @log_level("INFO")
# def load_data(): return "data loaded"
#
# @log_level("WARNING")
# def risky_op(): return "done"
#
# load_data()   → [INFO] Calling load_data
#                  [INFO] load_data completed
# risky_op()    → [WARNING] Calling risky_op
#                  [WARNING] risky_op completed

import functools

# YOUR CODE HERE


In [ ]:
# Exercise 7.3
# Write a decorator factory `throttle(calls, per_seconds)` that
# raises a RuntimeError if the function is called more than
# `calls` times within any `per_seconds` window.
#
# Hint: store timestamps of recent calls.
# Use time.time() and a list.
#
# @throttle(calls=3, per_seconds=1)
# def api_call(): return "ok"
#
# Calling api_call() 3 times should succeed.
# The 4th call (within the same second) should raise RuntimeError.

import functools
import time

# YOUR CODE HERE


## 8. Stacking Decorators

Multiple decorators can be **stacked** on a single function. They apply **bottom-up** during decoration, and **top-down** during execution.

```python
@decorator_A
@decorator_B
@decorator_C
def func(): ...

# Equivalent to:
func = decorator_A(decorator_B(decorator_C(func)))

# Execution order when func() is called:
# → decorator_A wrapper starts
#   → decorator_B wrapper starts
#     → decorator_C wrapper starts
#       → original func runs
#     ← decorator_C wrapper ends
#   ← decorator_B wrapper ends
# ← decorator_A wrapper ends
```

In [ ]:
# Example: Stacking order demonstrated
import functools

def bold(func):
    @functools.wraps(func)
    def wrapper(*a, **kw):
        return f"<b>{func(*a, **kw)}</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*a, **kw):
        return f"<i>{func(*a, **kw)}</i>"
    return wrapper

def underline(func):
    @functools.wraps(func)
    def wrapper(*a, **kw):
        return f"<u>{func(*a, **kw)}</u>"
    return wrapper

@bold       # applied 3rd (outermost)
@italic     # applied 2nd
@underline  # applied 1st (innermost)
def text(content):
    return content

# Execution: bold(italic(underline(text(content))))
print(text("Hello"))     # <b><i><u>Hello</u></i></b>

In [ ]:
# Example: Combining timer + log + validate in a real scenario
import functools
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        r = func(*args, **kwargs)
        print(f"  ⏱  {func.__name__} took {time.perf_counter()-t0:.4f}s")
        return r
    return wrapper

def announce(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"▶ Starting {func.__name__}")
        r = func(*args, **kwargs)
        print(f"✓ Finished {func.__name__}")
        return r
    return wrapper

@announce    # runs outermost
@timer       # runs in middle
def process_data(n):
    time.sleep(0.02)
    return sum(range(n))

result = process_data(1_000_000)
print(f"Result: {result}")

### Exercises — Stacking Decorators

In [ ]:
# Exercise 8.1
# Create two decorators:
#   - `strip_result`: strips leading/trailing whitespace from the return value
#   - `capitalize_result`: capitalizes the first letter of the return value
#
# Stack them on a function `get_input(text)` that just returns text.
# Test with:  get_input("  hello world  ")
#
# Try both orderings and observe the difference:
#   @strip_result + @capitalize_result  → "Hello world"
#   @capitalize_result + @strip_result  → "  hello world"
#   (why does order matter here?)

import functools

# YOUR CODE HERE


In [ ]:
# Exercise 8.2
# Build a mini web-framework-style decorator stack for a "route handler".
# Create three decorators:
#   - `authenticate`: prints "[AUTH] Checking credentials..." before the function
#   - `log_request`: prints "[LOG] Request received for: func_name" before the function
#   - `format_response`: wraps the return value as {"status": 200, "data": <result>}
#
# Stack all three on:
#   def get_user(user_id): return f"User #{user_id}"
#
# get_user(42) should print the auth and log messages and return the formatted dict.

import functools

# YOUR CODE HERE


In [ ]:
# Exercise 8.3
# Combine a decorator WITH ARGUMENTS + stacking:
#
# Create:
#   - `tag(html_tag)` — decorator factory that wraps the return value
#     in the given HTML tag: tag("p")(func) → "<p>result</p>"
#   - `add_class(css_class)` — decorator factory that adds a class attribute:
#     adds class="css_class" to the outermost tag (simple string replacement)
#
# Stack them to produce:
#   @add_class("highlight")
#   @tag("strong")
#   @tag("p")
#   def content(): return "Important message"
#
# Expected: '<p class="highlight"><strong>Important message</strong></p>'
# (Hint: add_class should replace the first '<' with '<tag class="...')

import functools

# YOUR CODE HERE


## 9. Real-World Decorator Patterns

Decorators power many everyday Python tools:

| Decorator | Where you've seen it |
|---|---|
| `@property` | Built-in Python |
| `@staticmethod`, `@classmethod` | Built-in Python |
| `@functools.lru_cache` | Standard library |
| `@app.route("/")` | Flask / FastAPI |
| `@pytest.fixture` | Testing |
| `@dataclass` | Standard library |

Below we implement simplified versions of two important patterns: **caching** and **access control**.

In [ ]:
# Example 1: Python's built-in @functools.lru_cache
import functools

@functools.lru_cache(maxsize=128)
def expensive_computation(n):
    """Simulates an expensive computation (e.g., DB query, API call)."""
    import time
    time.sleep(0.01)  # slow!
    return n ** 2

import time
start = time.perf_counter()
for _ in range(5):
    expensive_computation(42)  # only computes once; cached after that
print(f"5 calls took {time.perf_counter()-start:.4f}s")
print(expensive_computation.cache_info())

In [ ]:
# Example 2: Access control decorator
import functools

# Simulated "current user" system
CURRENT_USER = {"name": "Alice", "roles": ["admin", "user"]}

def require_role(role):
    """Only allow users who have the given role."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if role not in CURRENT_USER["roles"]:
                raise PermissionError(
                    f"Access denied: '{role}' role required, "
                    f"but user has {CURRENT_USER['roles']}"
                )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role("admin")
def delete_database():
    return "Database deleted!"

@require_role("superadmin")
def nuke_everything():
    return "Everything gone!"

print(delete_database())   # works — Alice is admin

try:
    nuke_everything()      # fails — Alice is not superadmin
except PermissionError as e:
    print(f"Caught: {e}")

In [ ]:
# Example 3: @property — a built-in decorator
# @property turns a method into a "smart attribute" with getter/setter logic.

class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

t = Temperature(100)
print(t.celsius)      # 100
print(t.fahrenheit)   # 212.0
t.celsius = 0
print(t.fahrenheit)   # 32.0

try:
    t.celsius = -300
except ValueError as e:
    print(f"Error: {e}")

### Exercises — Real-World Patterns

In [ ]:
# Exercise 9.1
# Write your OWN `memoize` decorator (no factory, no maxsize).
# It should cache all results in a dict keyed by the arguments.
#
# Apply it to a recursive `fib(n)` function and verify it's fast
# by timing fib(35) both with and without caching.
#
# Also expose the cache dict via: fib.cache

import functools
import time

# YOUR CODE HERE


In [ ]:
# Exercise 9.2
# Create a `BankAccount` class that uses @property to:
#   - Expose `balance` as a read-only computed property (returns self._balance)
#   - Expose `owner` with a getter and a setter that raises ValueError
#     if the new owner name is fewer than 2 characters
#
# Add methods `deposit(amount)` and `withdraw(amount)` that modify self._balance,
# with appropriate validation (no negative deposits, no overdraft).
#
# Test all paths including the error cases.

# YOUR CODE HERE


In [ ]:
# Exercise 9.3 — Grand Finale
# Combine generators AND decorators in one exercise.
#
# Part A — Generator:
#   Write a generator `number_stream(start=1)` that yields integers indefinitely.
#
# Part B — Decorator:
#   Write a decorator `trace(func)` that prints:
#     ">> Calling func_name(args)"
#   before calling, and:
#     "<< func_name returned: result"
#   after.
#
# Part C — Bring it together:
#   Write a function `take(n, iterable)` decorated with @trace.
#   It should return the first n items from any iterable as a list.
#
#   Use it to take the first 5 items from number_stream(10).
#   Expected trace output:
#     >> Calling take(5, <generator object ...>)
#     << take returned: [10, 11, 12, 13, 14]

import functools

# YOUR CODE HERE


## Summary Cheat Sheet

### Generators

```python
# Generator function — use yield instead of return
def my_gen(n):
    for i in range(n):
        yield i          # pauses here, resumes on next next()

# Consuming a generator
gen = my_gen(5)
next(gen)                # get next value
list(gen)                # consume all remaining values into a list
for x in my_gen(5): ... # iterate with a for loop

# Generator expression (lazy list comprehension)
gen_expr = (x**2 for x in range(10) if x % 2 == 0)
sum(gen_expr)            # consume with a built-in

# send() — two-way communication
def accumulator():
    total = 0
    while True:
        value = yield total  # receive AND send
        total += value

gen = accumulator()
next(gen)                # prime
gen.send(10)             # send a value in

# yield from — delegate to another iterable
def chain(a, b):
    yield from a
    yield from b
```

---

### Decorators

```python
import functools

# Basic decorator
def my_decorator(func):
    @functools.wraps(func)       # preserve name & docstring
    def wrapper(*args, **kwargs): # accept any arguments
        # --- before ---
        result = func(*args, **kwargs)
        # --- after ---
        return result
    return wrapper

@my_decorator
def greet(name): return f"Hi {name}"

# Decorator factory (decorator with arguments)
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)         # factory: returns the actual decorator
def say(msg): print(msg)

# Stacking — applied bottom-up, executed top-down
@decorator_A
@decorator_B
def func(): ...
# ≡ func = decorator_A(decorator_B(func))

# @property — turn a method into a smart attribute
class Foo:
    @property
    def value(self): return self._value
    @value.setter
    def value(self, v): self._value = v
```

---

**Key takeaways:**
- `yield` **pauses** a function and hands back a value; execution resumes from the same point on the next `next()`.
- Generators are **memory-efficient** — they produce values lazily, one at a time.
- Generator **pipelines** chain multiple generators without materialising intermediate lists.
- `send()` allows **two-way** communication with a running generator (coroutine pattern).
- A **decorator** wraps a function to add behaviour without changing its source code.
- Use `@functools.wraps` to preserve the wrapped function's identity.
- **Decorator factories** add a configuration layer by returning the decorator.
- **Stacking** applies decorators bottom-up; they execute outermost-first at call time.
